In [ ]:
import ir_datasets
import requests
import math
import pandas as pd

# ==========================================
# 1. إعدادات التقييم (تشغيل كامل بدون عينات)
# ==========================================
API_URL = "http://127.0.0.1:8009/search"
DATASET_API_NAME = "quora"
TEST_DATASET_NAME = "beir/quora/test"
FULL_DATASET_NAME = "beir/quora"
TOP_K = 10        

def get_dcg(rels):
    return sum([rel / math.log2(idx + 2) for idx, rel in enumerate(rels)])

def calculate_metrics(retrieved_texts, relevant_texts, k=10):
    retrieved_k = retrieved_texts[:k]
    rels = [1 if text in relevant_texts else 0 for text in retrieved_k]
    precision = sum(rels) / k
    recall = sum(rels) / len(relevant_texts) if relevant_texts else 0
    hits = 0; sum_precs = 0
    for i, rel in enumerate(rels):
        if rel:
            hits += 1; sum_precs += hits / (i + 1.0)
    ap = sum_precs / len(relevant_texts) if relevant_texts else 0
    dcg = get_dcg(rels)
    ideal_rels = sorted([1] * min(len(relevant_texts), k) + [0] * (k - min(len(relevant_texts), k)), reverse=True)
    idcg = get_dcg(ideal_rels)
    ndcg = dcg / idcg if idcg > 0 else 0
    return precision, recall, ap, ndcg

# ==========================================
# 2. تحميل البيانات الأصلية والـ Qrels
# ==========================================
print("📥 جاري تحميل قاعدة البيانات والـ Qrels بالكامل...\n")
full_dataset = ir_datasets.load(FULL_DATASET_NAME)
doc_texts = {doc.doc_id: doc.text.strip().lower() for doc in full_dataset.docs_iter()}

test_dataset = ir_datasets.load(TEST_DATASET_NAME)
qrels_dict = {}

for qrel in test_dataset.qrels_iter():
    if qrel.relevance > 0 and qrel.doc_id in doc_texts:
        if qrel.query_id not in qrels_dict: 
            qrels_dict[qrel.query_id] = set()
        qrels_dict[qrel.query_id].add(doc_texts[qrel.doc_id])

# استخراج جميع الاستعلامات التي تملك إجابات صحيحة
queries = [q for q in test_dataset.queries_iter() if q.query_id in qrels_dict]
total_queries = len(queries)

print("="*60)
print("📊 إحصائيات ملف الـ Qrels (جاهز للمناقشة):")
print(f"- إجمالي الاستعلامات (Queries) التي سيتم تقييم النظام بها: {total_queries} استعلام")
print("="*60 + "\n")

# ==========================================
# 3. تشغيل التقييم الفعلي على كامل البيانات
# ==========================================
algorithms = ["bm25", "hybrid_serial"]
results_summary = {algo: {"P": 0, "R": 0, "MAP": 0, "nDCG": 0} for algo in algorithms}

for algo in algorithms:
    print(f"\n⏳ جاري اختبار خوارزمية: {algo.upper()} على {total_queries} استعلام...")
    for i, query in enumerate(queries):
        payload = {"query": query.text, "dataset": DATASET_API_NAME, "algorithm": algo, "limit": TOP_K}
        try:
            response = requests.post(API_URL, json=payload)
            if response.status_code == 200:
                data = response.json()
                retrieved_texts = [res.get("text", res.get("content", "")).strip().lower() for res in data["results"]]
                p, r, ap, ndcg = calculate_metrics(retrieved_texts, qrels_dict[query.query_id], k=TOP_K)
                results_summary[algo]["P"] += p; results_summary[algo]["R"] += r
                results_summary[algo]["MAP"] += ap; results_summary[algo]["nDCG"] += ndcg
        except Exception as e:
            pass
            
        # طباعة مؤشر التقدم كل 100 استعلام لمعرفة حالة الخادم
        if (i + 1) % 100 == 0:
            print(f"   ✔️ تم إنجاز {i + 1} من {total_queries} استعلام...")
    
    # حساب المتوسط النهائي
    for metric in results_summary[algo]:
        results_summary[algo][metric] /= total_queries

# ==========================================
# 4. طباعة الجدول النهائي في النوت بوك
# ==========================================
print("\n✅ انتهى التقييم الشامل. النتائج النهائية:")
eval_data = {
    "Algorithm": ["BM25", "Hybrid Serial"],
    "Precision@10": [f"{results_summary['bm25']['P']*100:.2f}%", f"{results_summary['hybrid_serial']['P']*100:.2f}%"],
    "Recall": [f"{results_summary['bm25']['R']*100:.2f}%", f"{results_summary['hybrid_serial']['R']*100:.2f}%"],
    "MAP": [f"{results_summary['bm25']['MAP']*100:.2f}%", f"{results_summary['hybrid_serial']['MAP']*100:.2f}%"],
    "nDCG@10": [f"{results_summary['bm25']['nDCG']*100:.2f}%", f"{results_summary['hybrid_serial']['nDCG']*100:.2f}%"]
}
df_final = pd.DataFrame(eval_data)
display(df_final)

📥 جاري تحميل قاعدة البيانات والـ Qrels بالكامل...

📊 إحصائيات ملف الـ Qrels (جاهز للمناقشة):
- إجمالي الاستعلامات (Queries) التي سيتم تقييم النظام بها: 10000 استعلام


⏳ جاري اختبار خوارزمية: BM25 على 10000 استعلام...
   ✔️ تم إنجاز 100 من 10000 استعلام...
   ✔️ تم إنجاز 200 من 10000 استعلام...
   ✔️ تم إنجاز 300 من 10000 استعلام...
   ✔️ تم إنجاز 400 من 10000 استعلام...
   ✔️ تم إنجاز 500 من 10000 استعلام...
   ✔️ تم إنجاز 600 من 10000 استعلام...
   ✔️ تم إنجاز 700 من 10000 استعلام...
   ✔️ تم إنجاز 800 من 10000 استعلام...
   ✔️ تم إنجاز 900 من 10000 استعلام...
   ✔️ تم إنجاز 1000 من 10000 استعلام...
   ✔️ تم إنجاز 1100 من 10000 استعلام...
   ✔️ تم إنجاز 1200 من 10000 استعلام...
   ✔️ تم إنجاز 1300 من 10000 استعلام...
   ✔️ تم إنجاز 1400 من 10000 استعلام...
   ✔️ تم إنجاز 1500 من 10000 استعلام...
   ✔️ تم إنجاز 1600 من 10000 استعلام...
   ✔️ تم إنجاز 1700 من 10000 استعلام...
   ✔️ تم إنجاز 1800 من 10000 استعلام...
   ✔️ تم إنجاز 1900 من 10000 استعلام...
   ✔️ تم إنجاز 2000 من 10000 اس

,Algorithm,Precision@10,Recall,MAP,nDCG@10
0,BM25,11.60%,86.03%,71.26%,76.12%
1,Hybrid Serial,12.88%,92.63%,82.66%,86.24%
